In [68]:
# All imports

import os
import numpy as np
import pandas as pd
import gc 

def show_left_aligned(df):
    display(
        df.style
        .set_properties(**{"text-align": "left"})
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "left")]}
        ])
    )

pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)
pd.set_option('display.width', 1000)    

In [69]:

# Read the data and create summary table which will give an overview of the data in each dataset.
data_path = r".\python_project_aiml_logicmojo_dataset"

file_list = os.listdir(data_path)

summary = []

# print(file_list)

for file_name in file_list:
    file_path = os.path.join(data_path,file_name)
    df = pd.read_csv(file_path)
    # #Check for shape of the column 
    # print(f"Shape of file : {i} is {df.shape}")
    # #Check for the column list in each dataset.
    # print(f"Columns of file : {i} is {df.columns.to_list()}")
    # #check for duplicates in each dataset
    # dup_rec_count = df.duplicated().sum()
    # if dup_rec_count > 0:
    #     print(f"Shape of duplicate records for file: {i} is {df[df.duplicated(keep=False)].shape}")
    # # There are no duplicate records in any dataset, few records which shows that there are duplicate records in location.csv, we can ignore that as same zip code can be attched to muiple lat and lag values.
    # # Check for missing values 
    # for c in df.columns.to_list():
    #     miss_value = df[c].isna().sum()
    #     if miss_value > 0 :
    #         print(f"Number of missing valeues for file {i} of column {c} is {miss_value}")

    summary.append({
        "file_name": file_name,
        "n_rows": df.shape[0],
        "n_columns": df.shape[1],
        "columns": ", ".join(df.columns.tolist()),
        "duplicate_full_rows": df.duplicated().sum(),
        "n_missing_vals": df.isna().sum()[df.isna().sum() > 0].to_dict()
    })

summary_df = pd.DataFrame(summary)

show_left_aligned(summary_df)


,file_name,n_rows,n_columns,columns,duplicate_full_rows,n_missing_vals
0,category_translation.csv,71,2,"product_category_name, product_category_name_english",0,{}
1,customers.csv,99441,5,"customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state",0,{}
2,location.csv,1000163,5,"geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state",261831,{}
3,orders.csv,99441,8,"order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date",0,"{'order_approved_at': 160, 'order_delivered_carrier_date': 1783, 'order_delivered_customer_date': 2965}"
4,order_item.csv,112650,7,"order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value",0,{}
5,payments.csv,103886,5,"order_id, payment_sequential, payment_type, payment_installments, payment_value",0,{}
6,products.csv,32951,9,"product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm",0,"{'product_category_name': 610, 'product_name_lenght': 610, 'product_description_lenght': 610, 'product_photos_qty': 610, 'product_weight_g': 2, 'product_length_cm': 2, 'product_height_cm': 2, 'product_width_cm': 2}"
7,reviews.csv,99224,7,"review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp",0,"{'review_comment_title': 87656, 'review_comment_message': 58247}"
8,sellers.csv,3095,4,"seller_id, seller_zip_code_prefix, seller_city, seller_state",0,{}


# Load data to dataframes

In [70]:
# Load data to dataframes
customer_df = pd.read_csv(os.path.join(data_path, "customers.csv"))

orders_df = pd.read_csv(os.path.join(data_path, "orders.csv"))

order_item_df = pd.read_csv(os.path.join(data_path, "order_item.csv"))

products_df = pd.read_csv(os.path.join(data_path, "products.csv"))

cat_trans_df = pd.read_csv(os.path.join(data_path, "category_translation.csv"))

location_df = pd.read_csv(os.path.join(data_path, "location.csv"))

payments_df = pd.read_csv(os.path.join(data_path, "payments.csv"))

reviews_df = pd.read_csv(os.path.join(data_path,"reviews.csv"))

sellers_df = pd.read_csv(os.path.join(data_path, "sellers.csv"))


# Initial analysis on Customer data set

In [71]:
# Finding relationship between customer_unique_id and customer_id
customer_df.groupby("customer_unique_id")["customer_id"].nunique().reset_index(name="customer_id_count").loc[lambda df: df["customer_id_count"] > 1]

,customer_unique_id,customer_id_count
33,00172711b30d52eea8b313a7f2cced02,2
106,004288347e5e88a27ded2bb23747066c,2
124,004b45ec5c64187465168251cd1c9c2f,2
144,0058f300f57d7b93c477a131a59b36c3,2
249,00a39521eb40f7012db50455bf083460,2
...,...,...
95784,ff36be26206fffe1eb37afd54c70e18b,3
95810,ff44401d0d8f5b9c54a47374eb48c1b8,2
95916,ff8892f7c26aa0446da53d01b18df463,2
95934,ff922bdd6bafcdf99cb90d7f39cea5b3,3


# Initial analysis on Orders data set

In [72]:
# In orders tables we have the following columns with the number of null values associated to it 
# 'order_approved_at': 160, 'order_delivered_carrier_date': 1783, 'order_delivered_customer_date': 2965

#Analysis on order_approved_at missing values

print(orders_df[(orders_df["order_approved_at"].isna()) & (orders_df["order_status"] == "delivered")]) # these are the cases where the order_approved_at should not be NaN

print(orders_df.loc[(orders_df["order_approved_at"].isna()) & (orders_df["order_status"] != "delivered"), "order_status"].unique()) # Distinct order status ['canceled' 'created'], that means we can keep NaN for the cancelled orders

print(orders_df[(orders_df["order_approved_at"].isna()) & (orders_df["order_status"] == "created")]) # similarly for created oraders also we can keep the order_approved_at as NaN or we can pust some average date.


                               order_id                       customer_id order_status order_purchase_timestamp order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
5323   e04abd8149ef81b95221e88f6ed9ab6a  2127dc6603ac33544953ef05ec155771    delivered      2017-02-18 14:40:00               NaN          2017-02-23 12:04:47           2017-03-01 13:25:33           2017-03-17 00:00:00
16567  8a9adc69528e1001fc68dd0aaebbb54a  4c1ccc74e00993733742a3c786dc3c1f    delivered      2017-02-18 12:45:31               NaN          2017-02-23 09:01:52           2017-03-02 10:05:06           2017-03-21 00:00:00
19031  7013bcfc1c97fe719a7b5e05e61c12db  2941af76d38100e0f8740a374f1a5dc3    delivered      2017-02-18 13:29:47               NaN          2017-02-22 16:25:25           2017-03-01 08:07:38           2017-03-17 00:00:00
22663  5cf925b116421afa85ee25e99b4c34fb  29c35fc91fc13fb5073c8f30505d860d    delivered      2017-02-18 16:48:35             

In [73]:
# Analysis on order_delivered_carrier_date missing values
print(orders_df.loc[orders_df["order_approved_at"].isna(), "order_status"].unique())

# For delivered and created orders we can impute some values like average of the days after calculating what is the average number of days between order_approved_at and order_delivered_carrier_date.
# For cacelled orders we can keep the values as NaN as of now.

['canceled' 'delivered' 'created']


In [74]:
#Analysis on column order_delivered_customer_date
print(orders_df.loc[orders_df["order_delivered_customer_date"].isna(), "order_status"].unique())

orders_df.loc[(~orders_df["order_delivered_customer_date"].isna()) & (orders_df["order_status"]=='approved')]
# For delivered and created orders we can impute some values like average of the days after calculating what is the average number of days between order_approved_at and order_delivered_carrier_date.
# For cacelled orders we can keep the values as NaN as of now.
# If order_staus is invoiced,shipped,processing,unavailable,approved  then there are no rows with  order_delivered_customer_date as null.
# If order_staus is canceled there are 6 rows which still have order_delivered_customer_date as not null, but there are 619 rows where order_delivered_customer_date is null and order_status is cancellled, so I think we can drop those 6 rows.
# For those records where we have null values for order_delivered_customer_date and order_status as delivered, we have to impute the values with average days between order_delivered_carrier_date and order_delivered_customer_date


['invoiced' 'shipped' 'processing' 'unavailable' 'canceled' 'delivered'
 'created' 'approved']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


# Lets join the Customer and order table.

In [75]:
customer_orders_df = orders_df.merge(customer_df, how = "inner", on = "customer_id")
customer_orders_df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP
...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00,6359f309b166b0196dbf7ad2ac62bb5a,12209,sao jose dos campos,SP
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,da62f9e57a76d978d02ab5362c509660,11722,praia grande,SP
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,737520a9aad80b3fbbdad19b66b37b30,45920,nova vicosa,BA
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,5097a5312c8b157bb7be58ae360ef43c,28685,japuiba,RJ


# Add Ordered Items and seller details with customer_orders_df

In [76]:
sellers_df.columns

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')

In [77]:
customer_orders_seller_item_df = (customer_orders_df.merge(order_item_df, how = "inner", on = "order_id")
                                  .merge(sellers_df, how= "inner", on = "seller_id" ))

customer_orders_seller_item_df


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,9350,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,31570,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,14840,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,31842,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,8752,mogi das cruzes,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112645,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,da62f9e57a76d978d02ab5362c509660,11722,praia grande,SP,1,f1d4ce8c6dd66c47bbaa8c6781c2a923,1f9ab4708f3056ede07124aad39a2554,2018-02-12 13:10:37,174.90,20.10,17602,tupa,SP
112646,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,737520a9aad80b3fbbdad19b66b37b30,45920,nova vicosa,BA,1,b80910977a37536adeddd63663f916ad,d50d79cb34e38265a8649c383dcffd48,2017-09-05 15:04:16,205.99,65.02,8290,sao paulo,SP
112647,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,5097a5312c8b157bb7be58ae360ef43c,28685,japuiba,RJ,1,d1c427060a0f73f6b889a5c7c61f2ac4,a1043bafd471dff536d0c462352beb48,2018-01-12 21:36:21,179.99,40.59,37175,ilicinea,MG
112648,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,5097a5312c8b157bb7be58ae360ef43c,28685,japuiba,RJ,2,d1c427060a0f73f6b889a5c7c61f2ac4,a1043bafd471dff536d0c462352beb48,2018-01-12 21:36:21,179.99,40.59,37175,ilicinea,MG


# Analyis on products dataset

In [78]:
# print(products_df["product_category_name"].unique())
print(cat_trans_df.loc[cat_trans_df["product_category_name"] == "fashion_bolsas_e_acessorios"])



          product_category_name product_category_name_english
17  fashion_bolsas_e_acessorios      fashion_bags_accessories


In [79]:
{'product_category_name': 610, 'product_name_lenght': 610, 'product_description_lenght': 610, 'product_photos_qty': 610, 'product_weight_g': 2, 'product_length_cm': 2, 'product_height_cm': 2, 'product_width_cm': 2}

{'product_category_name': 610,
 'product_name_lenght': 610,
 'product_description_lenght': 610,
 'product_photos_qty': 610,
 'product_weight_g': 2,
 'product_length_cm': 2,
 'product_height_cm': 2,
 'product_width_cm': 2}

In [81]:
products_df.loc[products_df["product_category_name"].isna(), ["product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty"]].drop_duplicates()

,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty
105,NaN,NaN,NaN,NaN


In [82]:
# We will left join with products data set category_translation because for some of the product ids in products table we do not know the product category.
# As we are building the customer centric data sets so we will join the locvation data set with cusrtomer location and we dont want to miss any customer 

final_df = (customer_orders_seller_item_df.merge(products_df, how = "inner", on = "product_id")
            .merge(cat_trans_df, how = "left", on = "product_category_name")
            .merge(payments_df, how = "left", on = "order_id")
            .merge(location_df.loc[:,["geolocation_zip_code_prefix","geolocation_city", "geolocation_state"]].drop_duplicates(), how = "left", left_on=["customer_zip_code_prefix", "customer_city", "customer_state"]
                   , right_on= ["geolocation_zip_code_prefix", "geolocation_city", "geolocation_state"])
            .merge(reviews_df, how = "inner", on = "order_id")       
                   )
final_df 

# del customer_orders_seller_item_df

# gc.collect()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,payment_sequential,payment_type,payment_installments,payment_value,geolocation_zip_code_prefix,geolocation_city,geolocation_state,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,9350,maua,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,1.0,credit_card,1.0,18.12,3149.0,sao paulo,SP,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11 00:00:00,2017-10-12 03:43:48
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,9350,maua,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,3.0,voucher,1.0,2.00,3149.0,sao paulo,SP,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11 00:00:00,2017-10-12 03:43:48
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,9350,maua,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,2.0,voucher,1.0,18.59,3149.0,sao paulo,SP,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11 00:00:00,2017-10-12 03:43:48
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,31570,belo horizonte,SP,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,1.0,boleto,1.0,141.46,47813.0,barreiras,BA,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,14840,guariba,SP,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,1.0,credit_card,3.0,179.12,75265.0,vianopolis,GO,e73b67b67587f7644d5bd1a52deb1b01,5,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:

# Analysis on final data.

In [83]:
final_df.describe()

,customer_zip_code_prefix,order_item_id,price,freight_value,seller_zip_code_prefix,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,payment_sequential,payment_installments,payment_value,geolocation_zip_code_prefix,review_score
count,117332.000000,117332.000000,117332.000000,117332.000000,117332.000000,115637.000000,115637.000000,115637.000000,117312.000000,117312.000000,117312.000000,117312.000000,117329.000000,117329.000000,117329.000000,116964.000000,117332.000000
mean,35059.594978,1.194141,120.522417,20.027075,24452.248543,48.767635,785.809352,2.205497,2110.734656,30.254092,16.612461,23.071271,1.094452,2.940151,172.062565,34970.985260,4.031390
std,29849.293854,0.684241,182.942903,15.828114,27583.536615,10.033983,652.375747,1.717772,3785.084700,16.177472,13.452453,11.745779,0.731174,2.775370,265.388194,29835.162344,1.387994
min,1003.000000,1.000000,0.850000,0.000000,1001.000000,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000,1.000000,0.000000,0.000000,1003.000000,1.000000
25%,11250.000000,1.000000,39.900000,13.080000,6429.000000,42.000000,346.000000,1.000000,300.000000,18.000000,8.000000,15.000000,1.000000,1.000000,60.750000,11088.000000,4.000000
50%,24240.000000,1.000000,74.900000,16.280000,13660.000000,52.000000,600.000000,1.000000,700.000000,25.000000,13.000000,20.000000,1.000000,2.000000,108.100000,24230.000000,5.000000
75%,58770.000000,1.000000,134.900000,21.180000,28470.000000,57.000000,983.000000,3.000000,1800.000000,38.000000,20.000000,30.000000,1.000000,4.000000,189.060000,58280.000000,5.000000
max,99990.000000,21.000000,6735.000000,409.680000,99730.000000,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000,29.000000,24.000000,13664.080000,99990.000000,5.000000


In [84]:
print(final_df.info())
null_counts = final_df.isna().sum()
type(null_counts)
print(null_counts[null_counts>0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117332 entries, 0 to 117331
Data columns (total 43 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       117332 non-null  object 
 1   customer_id                    117332 non-null  object 
 2   order_status                   117332 non-null  object 
 3   order_purchase_timestamp       117332 non-null  object 
 4   order_approved_at              117317 non-null  object 
 5   order_delivered_carrier_date   116097 non-null  object 
 6   order_delivered_customer_date  114861 non-null  object 
 7   order_estimated_delivery_date  117332 non-null  object 
 8   customer_unique_id             117332 non-null  object 
 9   customer_zip_code_prefix       117332 non-null  int64  
 10  customer_city                  117332 non-null  object 
 11  customer_state                 117332 non-null  object 
 12  order_item_id                 

In [85]:
# Lets update the date columns from onject data type to date datatype 
columns_to_date = ["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "shipping_limit_date", "review_creation_date", "review_answer_timestamp"]

# print(final_df[columns_to_date].head())

# print("***************************************************************")

# print(final_df[columns_to_date].astype('datetime64[ns]').head())

# print("***************************************************************")

# print(final_df[columns_to_date].apply(pd.to_datetime).head())


# final_df[columns_to_date] = final_df[columns_to_date].astype('datetime64[ns]').head()

# final_df["order_purchase_timestamp"] = final_df["order_purchase_timestamp"].apply(pd.to_datetime)

final_df[columns_to_date] = final_df[columns_to_date].apply(
    lambda col: pd.to_datetime(col, errors="coerce")
)


In [86]:
date1 = pd.to_datetime("2017-02-18 14:40:00")
date2 = pd.to_datetime("2017-10-02 11:07:15")

print(round((date2-date1).total_seconds()/ (24 * 3600),2))

225.85


In [87]:
# final_df.loc[~(final_df["order_approved_at"].isna())]
# As all the missing values for order_approved_at are with order_status as delivered so we will compute the average days between order_purchase_timestamp and order_approved_at for order status as delivered.
# Create a new column order_approved_in_days. Impute the values for which the order_approved in days is null with average value.
# Let's create an other colum which is indicate if order has approval fopr it or not.
final_df["order_has_approval"] = final_df["order_approved_at"].notna().astype(int)
final_df["order_approved_in_days"] = ((final_df["order_approved_at"] - final_df["order_purchase_timestamp"]).dt.total_seconds()/(24 * 3600)).round(2)
# # final_df[(final_df["order_approved_in_days"].isna()) & (final_df["order_approved_at"].isna())]
# order_approved_mean = round(final_df.loc[final_df["order_approved_in_days"].notna(), "order_approved_in_days"].mean(),2)
# print(order_approved_mean)
# final_df["order_approved_in_days"] = final_df["order_approved_in_days"].fillna(order_approved_mean)
# print(final_df["order_approved_in_days"].isna().sum())

In [90]:
#lets impute the values for order_delivered_carrier_date
# print(final_df.loc[(final_df["order_delivered_carrier_date"].isna()), "order_status"].unique())

# print(final_df[final_df["order_delivered_carrier_date"].isna()].groupby("order_status")["order_status"].count())

# A better way is 
print(final_df.loc[final_df["order_delivered_carrier_date"].isna(), "order_status"].value_counts())

print(final_df.loc[final_df["order_delivered_carrier_date"].notna(), "order_status"].value_counts())

print(final_df.loc[((final_df["order_delivered_carrier_date"].notna()) & (final_df["order_delivered_customer_date"].notna())), "order_status"].value_counts())

# print(final_df.loc[(final_df["order_delivered_carrier_date"].notna()), "order_status"].unique())

# Lets createte a flag column called order_handed_to_carrier.

final_df["order_handed_to_carrier"] = final_df["order_delivered_carrier_date"].notna().astype(int)

final_df["order_shipped_in_days"] = ((final_df["order_delivered_customer_date"] - final_df["order_delivered_carrier_date"]).dt.total_seconds()/(24 * 3600)).round(2)




order_status
canceled       483
invoiced       370
processing     370
unavailable      7
approved         3
delivered        2
Name: count, dtype: int64
order_status
delivered    114860
shipped        1167
canceled         70
Name: count, dtype: int64
order_status
delivered    114853
canceled          7
Name: count, dtype: int64


In [91]:
final_df.iloc[:, -4:]

,order_has_approval,order_approved_in_days,order_handed_to_carrier,order_shipped_in_days
0,1,0.01,1,6.06
1,1,0.01,1,6.06
2,1,0.01,1,6.06
3,1,1.28,1,12.04
4,1,0.01,1,9.18
...,...,...,...,...
117327,1,0.01,1,20.76
117328,1,0.01,1,23.61
117329,1,0.01,1,13.33
117330,1,0.01,1,13.33


In [ ]:
# product_category_name              1695
# product_name_lenght                1695
# product_description_lenght         1695
# product_photos_qty                 1695
# product_weight_g                     20
# product_length_cm                    20
# product_height_cm                    20
# product_width_cm                     20
# product_category_name_english      1720


# product_category_name_english is having missing values, but its depended on product_category_name, because if product_category_name is missing that means product_category_name_english is also missing.

final_df.loc[(final_df["product_category_name"].isna()) & (final_df["product_category_name_english"].isna())]
final_df.loc[(final_df["product_category_name"].notna()) & (final_df["product_category_name_english"].isna())]

final_df.loc[final_df["order_id"] == "1d7542bb5262913fe0516f7943b69a58"]

# order_item_df.loc[order_item_df["order_id"] == "1d7542bb5262913fe0516f7943b69a58"]

# cat_trans_df.loc[cat_trans_df["product_category_name"] == "pc_gamer"]

products_df.loc[products_df["product_id"] == "6727051471a0fc4a0e7737b57bff2549"]

final_df["cat_missing_flag"] = final_df["product_category_name"].notna().astype("int")

final_df["english_cat_missing_flag"] = final_df["product_category_name_english"].notna().astype("int")

final_df.iloc[:, -6:]




,order_has_approval,order_approved_in_days,order_handed_to_carrier,order_shipped_in_days,cat_missing_flag,english_cat_missing_flag
0,1,0.01,1,6.06,1,1
1,1,0.01,1,6.06,1,1
2,1,0.01,1,6.06,1,1
3,1,1.28,1,12.04,1,1
4,1,0.01,1,9.18,1,1
...,...,...,...,...,...,...
117327,1,0.01,1,20.76,1,1
117328,1,0.01,1,23.61,1,1
117329,1,0.01,1,13.33,1,1
117330,1,0.01,1,13.33,1,1


# Bellow are the columns for which we can do some safe imputations.

In [114]:
final_df["product_name_lenght"] = final_df["product_name_lenght"].fillna(0)
final_df["product_description_lenght"] = final_df["product_description_lenght"].fillna(0)
final_df["product_photos_qty"] = final_df["product_photos_qty"].fillna(0)

final_df["product_weight_g"] = final_df["product_weight_g"].fillna(final_df["product_weight_g"].median())

In [115]:
final_df["payment_type"] = final_df["payment_type"].fillna("Unknown")
final_df["payment_installments"] = final_df["payment_installments"].fillna(1)
final_df["payment_value"] = final_df["payment_value"].fillna(0)
final_df["payment_sequential"] = final_df["payment_sequential"].fillna(0)

In [116]:
final_df["geo_missing_flag"] = (
    final_df["geolocation_zip_code_prefix"].isna()
).astype(int)

final_df["geolocation_zip_code_prefix"] = final_df["geolocation_zip_code_prefix"].fillna(-1)
final_df["geolocation_city"] = final_df["geolocation_city"].fillna("Unknown")
final_df["geolocation_state"] = final_df["geolocation_state"].fillna("Unknown")

In [117]:
final_df["has_review_comment"] = (
    final_df["review_comment_message"].notna()
).astype(int)

final_df["review_comment_title"] = final_df["review_comment_title"].fillna("")
final_df["review_comment_message"] = final_df["review_comment_message"].fillna("")

In [122]:
final_df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,payment_sequential,payment_type,payment_installments,payment_value,geolocation_zip_code_prefix,geolocation_city,geolocation_state,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,order_has_approval,order_approved_in_days,order_handed_to_carrier,order_shipped_in_days,cat_missing_flag,english_cat_missing_flag,geo_missing_flag,has_review_comment
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,9350,maua,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,1.0,credit_card,1.0,18.12,3149.0,sao paulo,SP,a54f0611adc9ed256b57ede6b6eb5114,4,,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,1,0.01,1,6.06,1,1,0,1
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,9350,maua,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,3.0,voucher,1.0,2.00,3149.0,sao paulo,SP,a54f0611adc9ed256b57ede6b6eb5114,4,,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,1,0.01,1,6.06,1,1,0,1
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,9350,maua,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,2.0,voucher,1.0,18.59,3149.0,sao paulo,SP,a54f0611adc9ed256b57ede6b6eb5114,4,,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,1,0.01,1,6.06,1,1,0,1
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,31570,belo horizonte,SP,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,1.0,boleto,1.0,141.46,47813.0,barreiras,BA,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,1,1.28,1,12.04,1,1,0,1
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,14840,guariba,SP,automotivo,46